# Pokémon Rojo Fuego Full + B10 — ROM privada para MyBoy

**Personal, experimental; no es una versión final.** Este cuaderno descarga el código público del proyecto y los sprites fijados, compila en una máquina privada de Colab y verifica la ROM contra el SHA-1 del CI de `e94824c`. No publica la ROM ni los gráficos en GitHub. No confiere derechos para redistribuir sprites o el juego.

Ejecuta primero la celda de compilación y, solo si aparece `ROM_VERIFICADA_CONTRA_CI`, ejecuta la celda de descarga. No necesitas conectar Google Drive ni pegar claves. Si Colab da error, conserva el mensaje exacto y no uses un archivo anterior.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess

REPO = "https://github.com/joseiturriaga25amz/pokefirered-europe.git"
BRANCH = "diagnostic/b10-full-integrated-candidate"
PIN = "e94824cf7afcf8062c383f6af72dc5c7cf37852f"
CI_SHA1 = "6fed3237225dc6146649d85c969b1e9da4352126"
OUTPUT = Path("/content/b10-myboy-privado/Pokemon_Rojo_Fuego_Full_B10_386.gba")
SOURCE = Path("/content/b10-full-source")

def cmd(*args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def read(*args, cwd=None):
    return subprocess.check_output(args, cwd=cwd, text=True).strip()

if SOURCE.exists() or OUTPUT.exists() or OUTPUT.with_suffix(".diagnostic.json").exists():
    raise RuntimeError("Ya hay una carpeta/ROM de una ejecucion anterior. Reinicia el entorno de Colab para evitar mezclar compilaciones.")

cmd("apt-get", "-o", "Acquire::Retries=3", "update")
cmd("apt-get", "-o", "Acquire::Retries=3", "install", "-y", "git", "make", "gcc", "pkg-config", "gcc-arm-none-eabi", "binutils-arm-none-eabi", "libpng-dev")
cmd("git", "clone", "--depth", "50", "--single-branch", "--branch", BRANCH, REPO, str(SOURCE))
if read("git", "branch", "--show-current", cwd=SOURCE) != BRANCH:
    raise RuntimeError("Rama inesperada")
cmd("git", "cat-file", "-e", PIN + "^{commit}", cwd=SOURCE)
cmd("git", "reset", "--hard", PIN, cwd=SOURCE)
if read("git", "rev-parse", "HEAD", cwd=SOURCE) != PIN:
    raise RuntimeError("Commit incorrecto")
cmd("python3", "tools/build_follower_diagnostic.py", "--self-test", cwd=SOURCE)
cmd("python3", "tools/build_follower_diagnostic.py", "--download-pinned", "--output", str(OUTPUT), cwd=SOURCE)
manifest = json.loads(OUTPUT.with_suffix(".diagnostic.json").read_text())
rom = OUTPUT.read_bytes()
if not (0 < len(rom) <= 32 * 1024 * 1024):
    raise RuntimeError("Tamano de ROM fuera de rango")
if manifest.get("project_git_sha") != PIN or manifest.get("national_species_count") != 386 or manifest.get("normal_shiny_variants") != 2:
    raise RuntimeError("Identidad del proyecto o cobertura incorrecta")
if manifest.get("source_feasible_species") != 386 or manifest.get("source_blocked_species") != 0:
    raise RuntimeError("Cobertura grafica 386/386 incompleta")
if manifest.get("rom_bytes") != len(rom) or manifest.get("rom_sha1") != hashlib.sha1(rom).hexdigest() or manifest.get("rom_sha256") != hashlib.sha256(rom).hexdigest():
    raise RuntimeError("La huella de la ROM no coincide con su manifiesto")
if manifest["rom_sha1"] != CI_SHA1:
    raise RuntimeError("La ROM no coincide con la compilacion privada auditada en CI; NO descargar ni probar esta version")
if (SOURCE / "pokefirered_modern_es.gba").exists() or (SOURCE / "graphics/object_events/pics/follower_import").exists():
    raise RuntimeError("Contaminacion del directorio fuente")
print("ROM_VERIFICADA_CONTRA_CI", "SHA-1:", manifest["rom_sha1"], "Bytes:", len(rom))
print("Archivo privado listo para descarga:", OUTPUT)
print("No hay prueba de MyBoy ni autorizacion de redistribucion de sprites.")


## Descargar en el teléfono
La siguiente celda solicita al navegador descargar **un archivo `.gba`**. En Android, abre el archivo descargado desde MyBoy. No compartas públicamente el archivo. Guarda el SHA-1 mostrado para identificar la ROM exacta.


In [ ]:
from google.colab import files
if not OUTPUT.is_file() or hashlib.sha1(OUTPUT.read_bytes()).hexdigest() != CI_SHA1:
    raise RuntimeError('No existe una ROM privada verificada; ejecuta primero la compilacion')
files.download(str(OUTPUT))
